In [0]:
# =====================================================================
# proceso / 05_load.py
# =====================================================================

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "retail_medallion")
catalogo = dbutils.widgets.get("catalogo")

superstore = spark.table(f"{catalogo}.silver.superstore_clean")
ecommerce = spark.table(f"{catalogo}.silver.ecommerce_daily_clean")

In [0]:
dim_product = superstore.select("product_id", "category", "sub_category").dropDuplicates(["product_id"])
dim_product.write.mode("overwrite").insertInto(f"{catalogo}.golden.dim_product")

dim_customer = superstore.select("customer_id", "customer_segment", "region").dropDuplicates(["customer_id"])
dim_customer.write.mode("overwrite").insertInto(f"{catalogo}.golden.dim_customer")

In [0]:
fact_sales = superstore.select(
    "order_id", "order_date", "customer_id", "product_id", "region",
    "sales_amount", "quantity", "discount_pct", "profit_amount", "profit_margin_pct",
)
fact_sales.write.mode("overwrite").insertInto(f"{catalogo}.golden.fact_sales")

In [0]:
agg_superstore = (
    superstore
    .withColumn("year_month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("year_month", "category", "region")
    .agg(
        F.round(F.sum("sales_amount"), 2).alias("total_sales"),
        F.round(F.sum("profit_amount"), 2).alias("total_profit"),
        F.round(F.avg("profit_margin_pct"), 4).alias("avg_profit_margin"),
        F.sum("quantity").alias("total_units"),
        F.countDistinct("order_id").alias("num_orders"),
    )
    .withColumn("source_system", F.lit("superstore"))
)

agg_ecommerce = (
    ecommerce
    .withColumn("year_month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("year_month", "category")
    .agg(
        F.round(F.sum("sales_amount"), 2).alias("total_sales"),
        F.round(F.sum("profit_amount"), 2).alias("total_profit"),
        F.round(F.avg("profit_margin_pct"), 4).alias("avg_profit_margin"),
        F.sum("units_sold").alias("total_units"),
        F.count("order_date").alias("num_orders"), 
    )
    .withColumn("region", F.lit(None).cast("string")) 
    .withColumn("source_system", F.lit("ecommerce"))
)

agg_sales = agg_superstore.unionByName(agg_ecommerce).select(
    "year_month", "category", "region", "source_system",
    "total_sales", "total_profit", "avg_profit_margin", "total_units", "num_orders",
)
agg_sales.write.mode("overwrite").insertInto(f"{catalogo}.golden.agg_sales_by_category_month")

In [0]:
agg_holiday_superstore = (
    superstore
    .groupBy("is_holiday", "holiday_name")
    .agg(
        F.round(F.sum("sales_amount"), 2).alias("total_sales"),
        F.round(F.sum("profit_amount"), 2).alias("total_profit"),
        F.countDistinct("order_id").alias("num_orders"),
    )
    .withColumn("source_system", F.lit("superstore"))
)

agg_holiday_ecommerce = (
    ecommerce
    .groupBy("is_holiday", "holiday_name")
    .agg(
        F.round(F.sum("sales_amount"), 2).alias("total_sales"),
        F.round(F.sum("profit_amount"), 2).alias("total_profit"),
        F.count("order_date").alias("num_orders"),
    )
    .withColumn("source_system", F.lit("ecommerce"))
)

agg_holiday = (
    agg_holiday_superstore.unionByName(agg_holiday_ecommerce)
    .withColumn("avg_order_value", F.round(F.col("total_sales") / F.col("num_orders"), 2))
    .select("is_holiday", "holiday_name", "source_system", "total_sales", "total_profit", "num_orders", "avg_order_value")
)
agg_holiday.write.mode("overwrite").insertInto(f"{catalogo}.golden.agg_sales_by_holiday")

In [0]:
marketing_roi = (
    ecommerce
    .withColumn("year_month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("year_month", "category")
    .agg(
        F.round(F.sum("sales_amount"), 2).alias("total_sales"),
        F.round(F.sum("marketing_spend"), 2).alias("total_marketing_spend"),
    )
    .withColumn("roi", F.round(F.col("total_sales") / F.col("total_marketing_spend"), 2))
    .select("year_month", "category", "total_sales", "total_marketing_spend", "roi")
)
marketing_roi.write.mode("overwrite").insertInto(f"{catalogo}.golden.ecommerce_marketing_roi")

print("Golden OK -> dim_product, dim_customer, fact_sales, agg_sales_by_category_month, agg_sales_by_holiday, ecommerce_marketing_roi")

In [0]:
jdbc_host = dbutils.secrets.get("retail-kv-scope", "sql-server-host")
jdbc_db = dbutils.secrets.get("retail-kv-scope", "sql-database-name")
jdbc_user = dbutils.secrets.get("retail-kv-scope", "sql-user")
jdbc_password = dbutils.secrets.get("retail-kv-scope", "sql-password")
jdbc_url = f"jdbc:sqlserver://{jdbc_host}:1433;database={jdbc_db};encrypt=true;trustServerCertificate=false;"

for table_name, df in [
    ("dim_product", dim_product),
    ("dim_customer", dim_customer),
    ("fact_sales", fact_sales),
    ("agg_sales_by_category_month", agg_sales),
    ("agg_sales_by_holiday", agg_holiday),
    ("ecommerce_marketing_roi", marketing_roi),
]:
    (
        df.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", f"golden.{table_name}")
        .option("user", jdbc_user)
        .option("password", jdbc_password)
        .mode("overwrite")
        .save()
    )
    print(f"Publicado en Azure SQL Database -> golden.{table_name}")